# Detecting Depression-Related Language in Tweets Using Machine Learning

This project classifies **tweet-level dataset labels**, not clinical diagnoses or people. We evaluate decision trees using top-ranked unigrams, bigrams, and trigrams, then investigate whether removing explicit depression terms changes performance. Feature selection occurs **only on the training partition**.

In [ ]:
import re
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, f1_score
)

RANDOM_STATE = 42
TOP_K_PER_CLASS = 5
TEST_SIZE = 0.20

## Data audit and split

First remove exact repeated tweet texts (after trimming surrounding whitespace). If the same text has conflicting labels, exclude **all** copies of that text rather than arbitrarily choosing a class. Splitting after de-duplication prevents exact repeated texts from appearing in both partitions.

In [ ]:
raw = pd.read_csv("sentiment_tweets3.csv")
assert {"message", "label"}.issubset(raw.columns), "Expected message and label columns"

df = raw[["message", "label"]].dropna().copy()
df["message"] = df["message"].astype(str).str.strip()
df = df[df["message"].ne("")].copy()
df["label"] = pd.to_numeric(df["label"], errors="raise").astype(int)
assert set(df["label"].unique()) == {0, 1}, "Expected binary labels 0 and 1"

print("Original rows:", len(raw))
print("Usable rows:", len(df))
print("Class distribution before de-duplication:")
print(df["label"].value_counts().sort_index())

conflicting = df.groupby("message")["label"].nunique()
ambiguous_texts = set(conflicting[conflicting > 1].index)
print("Identical texts with conflicting labels:", len(ambiguous_texts))
df = df[~df["message"].isin(ambiguous_texts)].drop_duplicates(subset=["message"]).reset_index(drop=True)
print("Rows after de-duplication:", len(df))
print("Class distribution after de-duplication:")
print(df["label"].value_counts().sort_index())

X_train, X_test, y_train, y_test = train_test_split(
    df["message"], df["label"], test_size=TEST_SIZE,
    random_state=RANDOM_STATE, stratify=df["label"]
)
assert set(X_train).isdisjoint(set(X_test)), "Train/test text overlap"
print("Train:", len(X_train), "Test:", len(X_test))

## Selecting and applying n-gram features

For each n, select the five most frequent n-grams among class 0 training tweets and the five most frequent among class 1 training tweets. The vocabulary is learned only from the training text and labels. The fitted vectorizer then transforms test text without ever accessing test labels. This also fixes the original notebook's inconsistent unigram treatment of neutral tweets and avoids substring matches.

In [ ]:
def prepare_features(train_text, test_text, train_labels, n, top_k=5):
    selector = CountVectorizer(
        ngram_range=(n, n), stop_words="english",
        lowercase=True, binary=False
    )
    train_counts = selector.fit_transform(train_text)
    terms = np.asarray(selector.get_feature_names_out())
    selected = []
    for cls in [0, 1]:
        mask = (np.asarray(train_labels) == cls)
        frequencies = np.asarray(train_counts[mask].sum(axis=0)).ravel()
        ranked = sorted(
            (i for i in np.flatnonzero(frequencies)),
            key=lambda i: (-frequencies[i], terms[i])
        )
        chosen = [terms[i] for i in ranked[:top_k]]
        print(f"{n}-gram features for class {cls}:", chosen)
        selected.extend(chosen)

    selected = list(dict.fromkeys(selected))  # remove overlap across classes
    if not selected:
        raise ValueError("No n-gram features available for this configuration")
    # Fix the vocabulary once from training, and apply it unchanged to test.
    transformer = CountVectorizer(
        vocabulary=selected, ngram_range=(n, n),
        stop_words="english", lowercase=True, binary=True
    )
    Xtr = transformer.transform(train_text)
    Xte = transformer.transform(test_text)
    return Xtr, Xte, selected


def evaluate(train_text, test_text, train_labels, test_labels, n, name):
    Xtr, Xte, feature_names = prepare_features(
        train_text, test_text, train_labels, n, TOP_K_PER_CLASS
    )
    model = DecisionTreeClassifier(random_state=RANDOM_STATE)
    model.fit(Xtr, train_labels)
    predicted = model.predict(Xte)
    metrics = {
        "experiment": name, "accuracy": accuracy_score(test_labels, predicted),
        "balanced_accuracy": balanced_accuracy_score(test_labels, predicted),
        "macro_f1": f1_score(test_labels, predicted, average="macro"),
        "n_features": len(feature_names)
    }
    print(f"\\n{name}")
    print(pd.Series(metrics))
    print(classification_report(test_labels, predicted, zero_division=0))
    print("Confusion matrix [rows: true 0, 1; columns: predicted 0, 1]:")
    print(confusion_matrix(test_labels, predicted, labels=[0, 1]))
    ConfusionMatrixDisplay.from_predictions(
        test_labels, predicted, labels=[0, 1], display_labels=["0", "1"]
    )
    plt.title(name)
    plt.show()
    return metrics, model, feature_names

## Unigrams, bigrams and trigrams — corrected evaluation

All configurations share the same held-out test partition. Scores below are produced when you run this notebook; they are not copied from the original evaluation, which used test labels during feature selection.

In [ ]:
results = []
models = {}
for n, label in [(1, "Unigrams"), (2, "Bigrams"), (3, "Trigrams")]:
    metrics, model, features = evaluate(X_train, X_test, y_train, y_test, n, label)
    results.append(metrics)
    models[label] = (model, features)

comparison = pd.DataFrame(results).set_index("experiment")
display(comparison.round(4))

## Sensitivity test: remove explicit depression terms

We strip **depression**, **depressed**, and **depressive** as complete words from both the training and test text, and learn all features afresh from the modified training text. This is an **ablation test** of the model's reliance on those words, not a claim that mentioning them is illegitimate evidence. The split and model specification are held constant.

In [ ]:
TERM_PATTERN = re.compile(r"\b(?:depression|depressed|depressive)\b", flags=re.IGNORECASE)

def remove_explicit_terms(text):
    return TERM_PATTERN.sub(" ", text)

train_ablated = X_train.map(remove_explicit_terms)
test_ablated = X_test.map(remove_explicit_terms)
print("Training tweets containing a removed term:", int(X_train.str.contains(TERM_PATTERN).sum()))
print("Test tweets containing a removed term:", int(X_test.str.contains(TERM_PATTERN).sum()))

ablation_metrics, ablation_model, ablation_features = evaluate(
    train_ablated, test_ablated, y_train, y_test, 1,
    "Unigrams without explicit depression terms"
)
comparison = pd.DataFrame(results + [ablation_metrics]).set_index("experiment")
display(comparison.round(4))
baseline = comparison.loc["Unigrams", "accuracy"]
ablated = comparison.loc["Unigrams without explicit depression terms", "accuracy"]
print(f"Accuracy change (ablation - baseline): {ablated - baseline:+.4f} ({(ablated - baseline)*100:+.2f} percentage points)")

## What we can and cannot conclude

The measured scores indicate how well models predict **the supplied tweet labels**. They do **not** establish that a tweet's author has clinically diagnosed depression. A statement such as "I have depression" may be relevant to a tweet-level label, but generic discussion of depression is not equivalent to personal experience.

An accuracy drop after word removal means explicit terms are important predictors under this dataset's sampling and labeling process; a small drop suggests that other n-gram patterns contribute. Neither result alone proves the model generalizes to new users or platforms. Check dataset provenance, near-duplicate content, class imbalance, and performance on independently annotated data before making broader claims.

Originally developed for the **Fundamentals of Network Analysis and Data Mining** course at the **University of Calgary**, taught by **Professor Emma Towlson**. The workflow here was subsequently revised to address evaluation leakage.